# Energy Comparison: Standard PC vs MCPC

This notebook compares **Standard Predictive Coding (PC)** and **Monte Carlo Predictive Coding (MCPC)**
using the same 1D generative regression task from `mcpc.py`.

In [ ]:
# Imports
import numpy as np
from scipy.stats import wasserstein_distance

import jax
import jax.numpy as jnp
from optax._src import base, combine, transform
import optax

import pcx as px
import pcx.predictive_coding as pxc
import pcx.nn as pxnn
import pcx.functional as pxf
import pcx.utils as pxu

# Reproducibility
np.random.seed(42)
px.RKG.seed(42)

## Energy Functions

In [ ]:
# Energy functions
def pc_energy(vode, rkg=px.RKG):
    # Standard PC energy (unit-variance Gaussian)
    e = vode.get("h") - vode.get("u")
    return 0.5 * (e * e)


def mcpc_energy(h_var: float):
    # MCPC energy scales the squared error by the target variance
    def energy(vode, rkg=px.RKG):
        e = vode.get("h") - vode.get("u")
        return 0.5 * (e * e) / h_var
    return energy

## Model, Optimizers, and Training

In [ ]:
# Stochastic gradient Langevin dynamics (SGLD) for MCPC state inference
def sgdld(
    learning_rate: base.ScalarOrSchedule,
    momentum: float = 0.0,
    h_var: float = 1.0,
    gamma: float = 0.0,
    seed: int = lambda: px.RKG(1)[0],
) -> base.GradientTransformation:
    eta = 2 * h_var * (1 - momentum) / learning_rate if momentum else 2 * h_var / learning_rate
    return combine.chain(
        transform.add_noise(eta, gamma, seed()),
        (transform.trace(decay=momentum) if momentum else base.identity()),
        transform.scale_by_learning_rate(learning_rate),
    )


# Model
class Model(pxc.EnergyModule):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        nm_layers: int,
        act_fn,
        energy_fn,
    ) -> None:
        super().__init__()
        self.act_fn = px.static(act_fn)

        self.layers = [pxnn.Linear(input_dim, hidden_dim)] + [
            pxnn.Linear(hidden_dim, hidden_dim, bias=False) for _ in range(nm_layers - 2)
        ] + [pxnn.Linear(hidden_dim, output_dim, bias=False)]

        self.vodes = [pxc.Vode(energy_fn=energy_fn) for _ in range(nm_layers)]
        self.vodes[-1].h.frozen = True

    def __call__(self, x, y=None):
        for v, l in zip(self.vodes[:-1], self.layers[:-1]):
            x = self.act_fn(v(l(x)))
        x = self.vodes[-1](self.layers[-1](x))
        if y is not None:
            self.vodes[-1].set("h", y)
        return self.vodes[-1].get("u")


# Forward + energy
@pxf.vmap(pxu.M(pxc.VodeParam | pxc.VodeParam.Cache).to((None, 0)), in_axes=(0, 0), out_axes=0)
def forward(x, y, *, model: Model):
    return model(x, y)


@pxf.vmap(
    pxu.M(pxc.VodeParam | pxc.VodeParam.Cache).to((None, 0)),
    in_axes=(0,),
    out_axes=(None, 0),
    axis_name="batch",
)
def energy(x, *, model: Model):
    y_ = model(x, None)
    return jax.lax.psum(model.energy(), "batch"), y_


# Train + eval
@pxf.jit(static_argnums=0)
def train_on_batch(T: int, x: jax.Array, y: jax.Array, *, model: Model, optim_w: pxu.Optim, optim_h: pxu.Optim):
    # Init step
    with pxu.step(model, (pxc.STATUS.INIT, None), clear_params=pxc.VodeParam.Cache):
        forward(x, y, model=model)
    optim_h.init(pxu.M_hasnot(pxc.VodeParam, frozen=True)(model))

    # Inference steps
    def h_step(i, x, *, model, optim_h):
        with pxu.step(model, clear_params=pxc.VodeParam.Cache):
            (_, _), g = pxf.value_and_grad(
                pxu.M_hasnot(pxc.VodeParam, frozen=True).to([False, True]),
                has_aux=True,
            )(energy)(x, model=model)
        optim_h.step(model, g["model"])
        return x, None

    pxf.scan(h_step, xs=jnp.arange(T))(x, model=model, optim_h=optim_h)
    optim_h.clear()

    # Learning step
    with pxu.step(model, clear_params=pxc.VodeParam.Cache):
        (_, _), g = pxf.value_and_grad(
            pxu.M(pxnn.LayerParam).to([False, True]),
            has_aux=True,
        )(energy)(x, model=model)
    optim_w.step(model, g["model"], scale_by=1.0 / x.shape[0])


@pxf.jit(static_argnums=0)
def eval_on_batch(T: int, x: jax.Array, *, model: Model, optim_h: pxu.Optim):
    # Init step
    with pxu.step(model, (pxc.STATUS.INIT, None), clear_params=pxc.VodeParam.Cache):
        forward(x, None, model=model)
    optim_h.init(pxu.M(pxc.VodeParam)(model))

    # Inference steps
    def h_step(i, x, *, model, optim_h):
        with pxu.step(model, clear_params=pxc.VodeParam.Cache):
            (_, _), g = pxf.value_and_grad(
                pxu.M_hasnot(pxc.VodeParam, frozen=True).to([False, True]),
                has_aux=True,
            )(energy)(x, model=model)
        optim_h.step(model, g["model"])
        return x, None

    pxf.scan(h_step, xs=jnp.arange(T))(x, model=model, optim_h=optim_h)
    optim_h.clear()


def train(dl, T: int, *, model: Model, optim_w: pxu.Optim, optim_h: pxu.Optim):
    model.vodes[-1].h.frozen = True
    for x, y in dl:
        train_on_batch(T, x, y, model=model, optim_w=optim_w, optim_h=optim_h)


def eval(dl, T: int, *, model: Model, optim_h: pxu.Optim):
    model.vodes[-1].h.frozen = False
    ys = []
    ys_ = []
    for x, y in dl:
        eval_on_batch(T, x, model=model, optim_h=optim_h)
        ys.append(y)
        ys_.append(model.vodes[-1].get("h"))
    model.vodes[-1].h.frozen = True

    ys = np.concatenate(ys, axis=0)
    ys_ = np.concatenate(ys_, axis=0)
    return wasserstein_distance(ys.squeeze(), ys_.squeeze()), ys_

## Run the Comparison

In [ ]:
# Configuration
batch_size = 32
lr_h = 1e-1
momentum = 0.5
h_var = 1.0
gamma = 0.0
lr_w = 1e-3
T = 100
T_eval = 100

# Target distribution
mean = 1.0
var = 5.0

# Generate data (inputs are zeros, outputs are Gaussian samples)
nm_elements = 10240
nm_elements_test = 1024
X = np.zeros((batch_size * (nm_elements // batch_size), 1))
y = np.random.randn(X.shape[0], 1) * np.sqrt(var) + mean
X_test = np.zeros((batch_size * (nm_elements_test // batch_size), 1))
y_test = np.random.randn(X_test.shape[0], 1) * np.sqrt(var) + mean

# Dataloaders
train_dl = list(zip(X.reshape(-1, batch_size, 1), y.reshape(-1, batch_size, 1)))
test_dl = tuple(zip(X_test.reshape(-1, batch_size, 1), y_test.reshape(-1, batch_size, 1)))

# Shared training budget
nm_epochs = 5120 // (nm_elements // batch_size)


def run_experiment(name, energy_fn, h_optim_fn):
    model = Model(
        input_dim=1,
        hidden_dim=1,
        output_dim=1,
        nm_layers=2,
        act_fn=lambda x: x,
        energy_fn=energy_fn,
    )

    # Optimizers
    with pxu.step(model, pxc.STATUS.INIT, clear_params=pxc.VodeParam.Cache):
        forward(jnp.zeros((batch_size, 1)), None, model=model)
        model.vodes[-1].h.frozen = True
        optim_h = pxu.Optim(h_optim_fn)
        optim_w = pxu.Optim(lambda: optax.adam(lr_w), pxu.M(pxnn.LayerParam)(model))
        model.vodes[-1].h.frozen = False

    model.vodes[-1].h.frozen = True
    w0, _ = eval(test_dl, T=T_eval, model=model, optim_h=optim_h)
    print(f"{name}: init Wasserstein {w0:.4f}")

    for _ in range(nm_epochs):
        train(train_dl, T=T, model=model, optim_w=optim_w, optim_h=optim_h)

    w, y_ = eval(test_dl, T=T_eval, model=model, optim_h=optim_h)
    print(f"{name}: final Wasserstein {w:.4f} | mean {y_.mean():.2f} | var {y_.var():.2f}")
    return model, y_


# Standard PC (deterministic state inference)
_ = run_experiment("Standard PC", pc_energy, lambda: optax.sgd(lr_h))

# MCPC (stochastic state inference)
_ = run_experiment("MCPC", mcpc_energy(h_var), sgdld(lr_h, momentum, h_var, gamma))